# Runnable 和 LCEL

## Runnable

Runnable是LangChain中的一个接口，它是Langchain核心组件的基础，多个Langchain核心组件都实现了Runnable接口。

- 模型调用 
- 输出解析器
- 检索器
- 工具调用

Langchain核心组件都实现了Runnable接口，因此它们的实例都具有`invoke`方法，Runnable接口强制要求所有LCEL组件都必须实现这样一组方法：
- invoke
- batch
- ainvoke
- abatch
- stream
- astream

因此我们可以通过管道符｜组合多个LCEL组件，实现复杂的任务，而程序也可以自动处理类型匹配和中间结果传递。

In [ ]:
import dotenv;
import os;
from langchain_openai import ChatOpenAI;
from langchain_core.prompts import ChatPromptTemplate;
from langchain_core.messages import SystemMessage, HumanMessage;
from langchain_core.output_parsers import StrOutputParser;



dotenv.load_dotenv();

OPENAI_API_KEY= os.getenv('OPENAI_API_KEY');
MODEL_NAME = os.getenv('MODEL_NAME')
BASE_URL = os.getenv('BASE_URL')


model = ChatOpenAI(
    model=os.environ['MODEL_NAME'],
    base_url=os.environ['BASE_URL'],
    api_key=os.environ['OPENAI_API_KEY'],
    temperature=0.5,
    max_tokens=1024,
)

template = ChatPromptTemplate.from_messages([
    ('system','假设你是一个语言专家'),
    ('human','请翻译这个单词{word}的英文为中文')
])
strOutputParser = StrOutputParser()

# 提示词模版实例可以调用invoke方法，返回一个消息列表
prompt = template.invoke({'word':'hello'})

# 模型实例可以调用invoke方法，返回结果
response = model.invoke(prompt)

# 输出解析器实例可以调用invoke方法，将模型输出解析为字符串
parsedOutput = strOutputParser.invoke(response.content)

print(parsedOutput)


"Hello" 的中文翻译是：**你好**

这是英语中最常用的问候语，相当于中文里的"你好"，用于见面时的打招呼或日常交流中。


以上代码如果使用Runnable，就可以将模型调用封装为一个可重复使用的组件。
也就是采用chain链的方式通过管道将多个组件组合起来，形成一个完整的流程。

In [17]:
chain = template | model | strOutputParser;
result = chain.invoke({'word':'cat'})
print(result)


作为语言专家，我来为您翻译这个单词：

**cat** 的中文翻译是：**猫**

这是英语中对家猫的称呼，在中文里对应的汉字就是"猫"。


## LCEL

LCEL的全称是Language Chain Expression Language。
它是一种用于描述和执行语言链的表达式语言。
它主要用于从现有的Runnable组件中构建新的Runnable的声明式方法。
LCEL 两个主要的组合原语是 RunnableSequence 和 RunnableParallel。许多其他组合原语可以被认为是这两个原语的变体。

### RunnableSequence 可运行序列

RunnableSequence 是一个组合原语，用于将多个Runnable组件按顺序组合起来，形成一个完整的流程。上一个组件的输出作为下一个组件的输入。

